# DA3METRIC-LARGE · Vehicle-Anchored Scale/Shift Correction + Full Evaluation

**Combined pipeline** unifying:
- `predict_intrinsics_da3.py` — per-sequence DA3 camera intrinsic prediction  
- `shift_scale_correction_v2.py` — vehicle size priors, weighted anchor logic  
- `DA_3_metric_analysis_premasked.ipynb` — DA3 inference, KITTI indexing, plots

---

## Pipeline
| Pass | What happens |
|------|-------------|
| **Pass 1** | Load pre-generated vehicle masks → run DA3 depth → predict per-sequence intrinsics → extract vehicle bbox anchors → RANSAC+WLS → global (scale, shift) |
| **Pass 2** | Apply global (scale, shift) to raw DA3 depth → evaluate all 5 scale variants vs LiDAR GT → write per-mask CSV |
| **Analysis** | 13 analysis plots + terminal summary table |

## Key Idea
The **vehicle-anchored** variant uses **no LiDAR ground truth** at test time.  
It estimates depth scale purely from:
- Predicted camera intrinsics (DA3 temporal joint method)
- Known vehicle physical dimensions (height ≈ 1.5 m, width ≈ 1.8 m)
- Bounding boxes of pre-segmented vehicle masks (SAM3 output from `inst_seg_notebook.ipynb`)

## Vehicle Size Priors
| Sub-category | Height | Width | Reliability |
|---|---|---|---|
| `vehicle_headlights` | 1.50 m ± 0.20 | 1.80 m ± 0.18 | 0.90 |
| `vehicle_taillights` | 1.45 m ± 0.22 | 1.80 m ± 0.18 | 0.92 |

## Cell 0 · GPU Selection & Imports

In [1]:
# ── Optional: select GPU ──────────────────────────────────────────────────────
# import os; os.environ["CUDA_VISIBLE_DEVICES"] = "1"

# ── Standard library ──────────────────────────────────────────────────────────
import csv, glob, json, logging, math, os, sys, warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
import torchvision.transforms as T

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D          # noqa: F401
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# ── Depth Anything 3 ──────────────────────────────────────────────────────────
# Install: git clone https://github.com/ByteDance-Seed/Depth-Anything-3
#          cd Depth-Anything-3 && pip install -e .
from depth_anything_3.api import DepthAnything3

print("Imports OK  |  CUDA available:", torch.cuda.is_available())

[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70
Imports OK  |  CUDA available: True


## Cell 1 · Logging

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("da3_va")
logging.getLogger("transformers").setLevel(logging.ERROR)

## Cell 2 · Configuration  *(edit paths here)*

All user-facing settings are collected here. Set `SKIP_PASS1 = True` if you
have already run Pass 1 and just want to re-use the cached scale/shift.
Set `SKIP_INFERENCE = True` to jump straight to analysis from an existing CSV.

In [3]:
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_DIR      = "/STORAGE/Arka/cache"

# ── Dataset ───────────────────────────────────────────────────────────────────
KITTI_ROOT     = "/STORAGE/Arka/Downloads/KITTI/kitti_eigen_split_dataset_small"
TEST_ROOT      = os.path.join(KITTI_ROOT, "test")
TRAIN_ROOT     = os.path.join(KITTI_ROOT, "train")
USE_TRAIN      = True

# ── Pre-generated mask store (output of inst_seg_notebook.ipynb) ──────────────
#    Layout (already inside the 'test' split folder):
#      MASK_ROOT_TEST|TRAIN / <date> / <drive> / image_02/data / <frame_stem> /
MASK_ROOT_TEST  = "/STORAGE/Arka/a_MDE_Code/1_Anaysis/Output/test"
MASK_ROOT_TRAIN = "/STORAGE/Arka/a_MDE_Code/1_Anaysis/Output/train"

# ── Depth clipping ────────────────────────────────────────────────────────────
MIN_DEPTH      = 1e-3
MAX_DEPTH      = 80.0

# ── KITTI default focal length (for DA3 metric conversion formula) ────────────
FOCAL_LENGTH_X = 721.5577

# ── Mask confidence gate ──────────────────────────────────────────────────────
MIN_MASK_SCORE = 0.50

# ── DA3 model identifiers ─────────────────────────────────────────────────────
DEPTH_MODEL_ID   = "depth-anything/DA3METRIC-LARGE"
INTRIN_MODEL_ID  = "depth-anything/DA3NESTED-GIANT-LARGE"

# ── Intrinsic prediction parameters ──────────────────────────────────────────
INTRIN_CHUNK_SIZE  = 20
INTRIN_PROCESS_RES = 504

# ── RANSAC parameters ─────────────────────────────────────────────────────────
RANSAC_ITERATIONS  = 1000
RANSAC_INLIER_THR  = 2.5    # metres — loosened to improve inlier rate on full-vehicle masks
MIN_ANCHORS_FOR_FIT = 4
MIN_PIXEL_DIM       = 8

# ── Output ────────────────────────────────────────────────────────────────────
MAIN_OUTPUT_DIR = "/STORAGE/Arka/a_MDE_Code/2_Corrections/Output/DA3_VehicleAnchored_V5_final"
OUTPUT_CSV_PATH = os.path.join(MAIN_OUTPUT_DIR, "DA3_VA_results.csv")
ANCHORS_CSV     = os.path.join(MAIN_OUTPUT_DIR, "vehicle_anchors.csv")
INTRINSICS_CSV  = os.path.join(MAIN_OUTPUT_DIR, "sequence_intrinsics.csv")
PLOT_DIR        = os.path.join(MAIN_OUTPUT_DIR, "plots")
SCALE_CACHE     = os.path.join(MAIN_OUTPUT_DIR, "global_scale_shift.json")
SAVE_NPY        = True

# ── Run control ───────────────────────────────────────────────────────────────
START_INDEX     = 0
SKIP_PASS1      = False   # True → reuse SCALE_CACHE from a previous run
SKIP_INFERENCE  = False   # True → skip Pass 2, go straight to analysis
SKIP_PLOTS      = False

os.makedirs(MAIN_OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)
log.info("Configuration loaded. DEVICE=%s", DEVICE)
print("Configuration loaded.")

21:07:36  INFO      Configuration loaded. DEVICE=cuda


Configuration loaded.


## Cell 3 · Vehicle Size Priors

Physical dimensions derived from `shift_scale_correction_v2.py → EXTENDED_PRIORS`.

| Sub-category | Real height | height_std | Real width | width_std | Reliability |
|---|---|---|---|---|---|
| `vehicle_headlights` | 1.50 m | 0.20 | 1.80 m | 0.18 | 0.90 |
| `vehicle_taillights` | 1.45 m | 0.22 | 1.80 m | 0.18 | 0.92 |

> **Note on "length" dimension:** Estimating vehicle length (front-to-rear distance)
> from a single image requires matching front- and rear-facing masks from different
> frames or using instance tracking. This is left as a future extension; height and
> width priors already provide two independent depth anchors per instance which are
> sufficient for scale/shift estimation.

In [4]:
@dataclass
class VehiclePrior:
    sub_cat:            str
    real_height:        Optional[float]
    real_width:         Optional[float]
    height_std:         float = 0.20
    width_std:          float = 0.20
    anchor_reliability: float = 1.0

    @property
    def height_weight(self) -> float:
        return (self.anchor_reliability / self.height_std ** 2
                if self.real_height else 0.0)

    @property
    def width_weight(self) -> float:
        return (self.anchor_reliability / self.width_std ** 2
                if self.real_width else 0.0)


# Priors are for the FULL VEHICLE bounding box (mask covers whole car).
# vehicle_headlights = front-facing vehicle; vehicle_taillights = rear-facing.
# Height is always meaningful (top of roof to road-level).
# Width is only meaningful when the vehicle is roughly frontal/rear-on
# (aspect ratio check in extract_vehicle_anchors disables width for side-on views).
# Tighter stds → higher weight per anchor → better RANSAC discrimination.
VEHICLE_PRIORS: Dict[str, VehiclePrior] = {
    "vehicle_headlights": VehiclePrior(
        sub_cat="vehicle_headlights",
        real_height=1.50, height_std=0.12,   # cars: 1.38–1.62 m (sedan/SUV range)
        real_width=1.82,  width_std=0.12,    # cars: 1.70–1.94 m
        anchor_reliability=0.90,
    ),
    "vehicle_taillights": VehiclePrior(
        sub_cat="vehicle_taillights",
        real_height=1.50, height_std=0.12,
        real_width=1.82,  width_std=0.12,
        anchor_reliability=0.92,
    ),
}
print("Vehicle priors:", list(VEHICLE_PRIORS.keys()))

Vehicle priors: ['vehicle_headlights', 'vehicle_taillights']


## Cell 4 · Camera Intrinsics  *(from `predict_intrinsics_da3.py`)*

DA3NESTED-GIANT-LARGE is used to predict per-sequence camera intrinsics via
temporal joint estimation (the same sliding-window approach used in
`predict_intrinsics_da3.py`).  Intrinsics are predicted **once per drive** and
cached in `_sequence_intrin_cache`, then reused for every frame in that drive.

The depth formula for a vehicle bounding box `(x1, y1, x2, y2)`:

$$d_h = \frac{f_y \cdot H_{\mathrm{real}}}{y_2 - y_1}, \quad
  d_w = \frac{f_x \cdot W_{\mathrm{real}}}{x_2 - x_1}$$

In [5]:
@dataclass
class CameraIntrinsics:
    fx:     float
    fy:     float
    cx:     float
    cy:     float
    fx_std: float = 0.0
    fy_std: float = 0.0
    source: str   = "assumed"

    @property
    def relative_uncertainty(self) -> float:
        return self.fx_std / (self.fx + 1e-6)

    @property
    def quality_weight(self) -> float:
        if self.source == "assumed_fov":
            return 0.60
        rel = self.relative_uncertainty
        return float(np.clip(1.0 - 10.0 * rel, 0.20, 1.0))

    def to_dict(self) -> dict:
        return dict(fx=self.fx, fy=self.fy, cx=self.cx, cy=self.cy,
                    fx_std=self.fx_std, fy_std=self.fy_std,
                    source=self.source, quality_weight=self.quality_weight)


_intrin_model_cache = None
_sequence_intrin_cache: Dict[str, CameraIntrinsics] = {}


def _load_intrin_model():
    global _intrin_model_cache
    if _intrin_model_cache is not None:
        return _intrin_model_cache
    log.info("Loading DA3 intrinsic model: %s …", INTRIN_MODEL_ID)
    m = DepthAnything3.from_pretrained(INTRIN_MODEL_ID, cache_dir=CACHE_DIR)
    m = m.to(DEVICE).eval()
    _intrin_model_cache = m
    log.info("DA3 intrinsic model loaded.")
    return m


def _fallback_intrinsics(W: int, H: int, fov_deg: float = 65.0) -> CameraIntrinsics:
    fov_rad = np.radians(fov_deg)
    fx = (W / 2.0) / np.tan(fov_rad / 2.0)
    return CameraIntrinsics(fx=fx, fy=fx, cx=W / 2.0, cy=H / 2.0,
                             fx_std=fx * 0.10, fy_std=fx * 0.10,
                             source="assumed_fov")


def predict_sequence_intrinsics(image_paths: List[str],
                                 image_wh: Tuple[int, int]) -> CameraIntrinsics:
    W, H  = image_wh
    model = _load_intrin_model()
    chunks = [image_paths[i: i + INTRIN_CHUNK_SIZE]
              for i in range(0, len(image_paths), INTRIN_CHUNK_SIZE)]
    K_chunks: List[np.ndarray] = []
    for chunk in chunks:
        try:
            with torch.no_grad():
                pred = model.inference(
                    image=chunk,
                    process_res=INTRIN_PROCESS_RES,
                    process_res_method="upper_bound_resize",
                )
            K = pred.intrinsics
            if hasattr(K, "cpu"):
                K = K.cpu().numpy()
            K_chunks.append(np.asarray(K, dtype=np.float64))
        except Exception as exc:
            log.warning("Intrinsic chunk failed: %s", exc)

    if not K_chunks:
        log.warning("All intrinsic chunks failed — using FOV fallback.")
        return _fallback_intrinsics(W, H)

    all_K = np.concatenate(K_chunks, axis=0)
    K_avg = all_K.mean(axis=0)
    K_std = all_K.std(axis=0)
    
    # FIX: DA3 resizes isotropically. The scale factor must be uniform.
    scale_factor = max(W, H) / INTRIN_PROCESS_RES

    scale_x = W / INTRIN_PROCESS_RES
    scale_y = H / INTRIN_PROCESS_RES
    K_avg[0, 0] *= scale_factor;  K_avg[1, 1] *= scale_factor
    K_avg[0, 2] *= scale_factor;  K_avg[1, 2] *= scale_factor
    K_std[0, 0] *= scale_factor;  K_std[1, 1] *= scale_factor

    cam = CameraIntrinsics(
        fx=float(K_avg[0, 0]), fy=float(K_avg[1, 1]),
        cx=float(K_avg[0, 2]), cy=float(K_avg[1, 2]),
        fx_std=float(K_std[0, 0]), fy_std=float(K_std[1, 1]),
        source="da3_temporal",
    )
    log.info("Intrinsics: fx=%.1f±%.1f  fy=%.1f±%.1f  quality=%.2f",
             cam.fx, cam.fx_std, cam.fy, cam.fy_std, cam.quality_weight)
    return cam


def get_sequence_intrinsics(seq_name: str,
                             seq_image_paths: List[str],
                             image_wh: Tuple[int, int]) -> CameraIntrinsics:
    if seq_name in _sequence_intrin_cache:
        return _sequence_intrin_cache[seq_name]
    log.info("Predicting intrinsics for: %s", seq_name)
    cam = predict_sequence_intrinsics(seq_image_paths, image_wh)
    _sequence_intrin_cache[seq_name] = cam
    return cam

print("Camera intrinsic helpers defined.")

Camera intrinsic helpers defined.


## Cell 5 · DA3METRIC-LARGE Depth Inference

Raw output is converted to orthographic metric depth (camera Z-axis, metres) via:

$$D_{\mathrm{metric}} = \frac{f_x^{\mathrm{KITTI}} \cdot D_{\mathrm{raw}}}{300}$$

Any systematic bias introduced by using the fixed KITTI focal length (rather than
the true per-sequence focal) is corrected by the vehicle-anchored scale/shift.

In [6]:
_da3_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])

_depth_model_cache = None


def load_depth_model():
    global _depth_model_cache
    if _depth_model_cache is not None:
        return _depth_model_cache
    log.info("Loading DA3METRIC-LARGE (%s) …", DEPTH_MODEL_ID)
    mdl = DepthAnything3.from_pretrained(
        DEPTH_MODEL_ID, cache_dir=CACHE_DIR
    ).to(DEVICE).eval()
    _depth_model_cache = mdl
    log.info("DA3 depth model ready on %s.", DEVICE)
    return mdl


def run_da3_depth(mdl, im_np: np.ndarray) -> np.ndarray:
    orig_h, orig_w = im_np.shape[:2]
    new_h = int(round(orig_h / 14) * 14)
    new_w = int(round(orig_w / 14) * 14)
    img_pil = Image.fromarray(im_np).resize((new_w, new_h), Image.BICUBIC)
    t_img   = _da3_transform(img_pil)
    inp     = t_img.unsqueeze(0).unsqueeze(1).to(DEVICE)
    with torch.no_grad():
        preds = mdl(inp, export_feat_layers=[])
        if isinstance(preds, dict):
            depth = preds["depth"]
        elif isinstance(preds, (tuple, list)):
            depth = preds[0]
        else:
            depth = preds
        if isinstance(depth, torch.Tensor):
            if depth.dim() == 3:
                depth = depth.unsqueeze(1)
            depth = (FOCAL_LENGTH_X * depth) / 300.0
            depth = F.interpolate(depth, size=(orig_h, orig_w),
                                   mode="bilinear", align_corners=False)
    return depth.squeeze().cpu().numpy().astype(np.float32)

print("DA3 depth model helpers defined.")

DA3 depth model helpers defined.


## Cell 6 · Pre-Generated Mask Loading  *(replaces SAM3 inference)*

Masks were generated by `inst_seg_notebook.ipynb` and stored under `MASK_ROOT`.  
The directory tree mirrors the KITTI layout exactly.

In [7]:
def get_mask_dir(rgb_path: str) -> str:
    is_test      = "/test/" in rgb_path
    split_root   = TEST_ROOT      if is_test else TRAIN_ROOT
    mask_root    = MASK_ROOT_TEST if is_test else MASK_ROOT_TRAIN
    frame_stem   = os.path.splitext(os.path.basename(rgb_path))[0]
    rel_data_dir = os.path.relpath(os.path.dirname(rgb_path), split_root)
    return os.path.join(mask_root, rel_data_dir, frame_stem)


def load_masks_for_frame(mask_dir: str,
                           super_cat_filter: Optional[str] = None,
                           sub_cat_filter:   Optional[str] = None) -> List[dict]:
    mask_dir_path = Path(mask_dir)
    if not mask_dir_path.exists():
        return []
    frame_stem    = mask_dir_path.name
    manifest_path = mask_dir_path / f"{frame_stem}_manifest.json"

    if manifest_path.exists():
        with open(manifest_path, "r") as fh:
            records = json.load(fh)
        instances = []
        for rec in records:
            if super_cat_filter and rec["super_cat"] != super_cat_filter:
                continue
            if sub_cat_filter   and rec["sub_cat"]   != sub_cat_filter:
                continue
            mask_file = mask_dir_path / rec["mask_file"]
            if not mask_file.exists():
                continue
            mask_np = np.array(Image.open(mask_file).convert("L")) > 127
            instances.append({
                "uid":       rec["uid"],
                "local_idx": rec["local_idx"],
                "super_cat": rec["super_cat"],
                "sub_cat":   rec["sub_cat"],
                "score":     float(rec["score"]),
                "box":       rec.get("box"),
                "mask":      mask_np,
                "path":      mask_file,
            })
        return instances

    # Fallback: glob + filename parsing
    instances = []
    for mask_file in sorted(mask_dir_path.glob("*_mask.png")):
        stem   = mask_file.stem[:-5]
        if not stem.startswith(frame_stem + "_"):
            continue
        suffix = stem[len(frame_stem) + 1:]
        parts  = suffix.rsplit("_", 3)
        if len(parts) != 4:
            continue
        cat_str, idx_str, uid_str, score_str = parts
        cat_parts = cat_str.split("_", 1)
        if len(cat_parts) != 2:
            continue
        super_cat, sub_cat = cat_parts
        if super_cat_filter and super_cat != super_cat_filter:
            continue
        if sub_cat_filter   and sub_cat   != sub_cat_filter:
            continue
        try:
            local_idx = int(idx_str); score = float(score_str)
        except ValueError:
            continue
        mask_np = np.array(Image.open(mask_file).convert("L")) > 127
        instances.append({"uid": uid_str, "local_idx": local_idx,
                           "super_cat": super_cat, "sub_cat": sub_cat,
                           "score": score, "box": None,
                           "mask": mask_np, "path": mask_file})
    instances.sort(key=lambda x: x["local_idx"])
    return instances

print("Mask loading helpers defined.")

Mask loading helpers defined.


## Cell 7 · KITTI Dataset Indexing

In [8]:
def index_kitti(use_train: bool = False) -> pd.DataFrame:
    rows = []
    def _collect(split_root: str, tag: str) -> None:
        pattern  = os.path.join(split_root, "*", "*_sync",
                                "proj_depth", "groundtruth", "image_02", "*.png")
        gt_paths = glob.glob(pattern)
        for gt_path in tqdm(gt_paths, desc=f"Indexing {tag}"):
            parts            = gt_path.split(os.sep)
            date, drive, fname = parts[-6], parts[-5], parts[-1]
            rgb_path = os.path.join(split_root, date, drive, "image_02", "data", fname)
            if not os.path.exists(rgb_path):
                continue
            rows.append(dict(folder=tag, sequence=date, drive=drive,
                             filename=fname, gt_path=gt_path, rgb_path=rgb_path))
    _collect(TEST_ROOT, "Test")
    if use_train:
        _collect(TRAIN_ROOT, "Train")
    df = pd.DataFrame(rows).reset_index(drop=True)
    log.info("Dataset indexed: %d frames, %d drives.", len(df), df["drive"].nunique())
    return df


def get_sequence_image_paths(drive_root: str) -> List[str]:
    return sorted(glob.glob(os.path.join(drive_root, "image_02", "data", "*.png")))

print("KITTI indexing helpers defined.")

KITTI indexing helpers defined.


In [9]:
# ── DEBUG: verify mask paths are resolving correctly ─────────────────────────
import os
from pathlib import Path

val_list_debug = index_kitti(use_train=USE_TRAIN)
sample = val_list_debug.iloc[0]
rgb_path = sample["rgb_path"]

frame_stem   = os.path.splitext(os.path.basename(rgb_path))[0]
split_root   = TEST_ROOT if "/test/" in rgb_path else TRAIN_ROOT
rel_data_dir = os.path.relpath(os.path.dirname(rgb_path), split_root)
mask_root    = MASK_ROOT_TEST if "/test/" in rgb_path else MASK_ROOT_TRAIN
mask_dir     = os.path.join(mask_root, rel_data_dir, frame_stem)

print(f"Sample rgb_path    : {rgb_path}")
print(f"split_root         : {split_root}")
print(f"rel_data_dir       : {rel_data_dir}")
print(f"Expected mask_dir  : {mask_dir}")
print(f"mask_dir exists    : {Path(mask_dir).exists()}")

# Show MASK_ROOT layout to compare
print(f"\nMASK_ROOT_TEST top-level contents:")
try:
    for entry in sorted(Path(MASK_ROOT_TEST).iterdir())[:5]:
        print(f"  {entry.name}/")
        for sub in sorted(entry.iterdir())[:3]:
            print(f"    {sub.name}/")
            for subsub in sorted(sub.iterdir())[:2]:
                print(f"      {subsub.name}/")
except Exception as e:
    print(f"  Could not list MASK_ROOT: {e}")


Indexing Test:   0%|          | 0/652 [00:00<?, ?it/s]

Indexing Train:   0%|          | 0/96 [00:00<?, ?it/s]

21:07:36  INFO      Dataset indexed: 748 frames, 29 drives.


Sample rgb_path    : /STORAGE/Arka/Downloads/KITTI/kitti_eigen_split_dataset_small/test/2011_09_26/2011_09_26_drive_0056_sync/image_02/data/0000000286.png
split_root         : /STORAGE/Arka/Downloads/KITTI/kitti_eigen_split_dataset_small/test
rel_data_dir       : 2011_09_26/2011_09_26_drive_0056_sync/image_02/data
Expected mask_dir  : /STORAGE/Arka/a_MDE_Code/1_Anaysis/Output/test/2011_09_26/2011_09_26_drive_0056_sync/image_02/data/0000000286
mask_dir exists    : True

MASK_ROOT_TEST top-level contents:
  2011_09_26/
    2011_09_26_drive_0002_sync/
      image_02/
    2011_09_26_drive_0009_sync/
      image_02/
    2011_09_26_drive_0013_sync/
      image_02/
  2011_09_28/
    2011_09_28_drive_0002_sync/
      image_02/
  2011_09_29/
    2011_09_29_drive_0071_sync/
      image_02/
  2011_09_30/
    2011_09_30_drive_0016_sync/
      image_02/
    2011_09_30_drive_0018_sync/
      image_02/
    2011_09_30_drive_0027_sync/
      image_02/
  2011_10_03/
    2011_10_03_drive_0027_sync/
     

## Cell 8 · Vehicle Anchor Extraction

For each pre-generated vehicle mask:

1. Retrieve bounding box `(x1, y1, x2, y2)` from the manifest JSON (or compute
   from mask pixels as fallback).
2. Compute `pixel_height = y2 − y1`, `pixel_width = x2 − x1`.
3. Estimate metric depth:
   - From height: $d_h = f_y \cdot H_{\mathrm{real}} / (y_2 - y_1)$
   - From width:  $d_w = f_x \cdot W_{\mathrm{real}} / (x_2 - x_1)$
4. Consistency check: if $d_h / d_w > 1.3$ use the higher-weight estimate only.
5. Compute affine depth $a_i = \mathrm{median}(D_{\mathrm{raw}}[\mathrm{mask}])$.
6. Return anchor: $(a_i,\; d_i,\; w_i)$ where $w_i = r_{\mathrm{prior}} \cdot q_{\mathrm{intrin}} \cdot s_{\mathrm{SAM3}} \cdot \Sigma_j (1/\sigma_j^2)$.

In [10]:
@dataclass
class VehicleAnchor:
    img_id:         int
    mask_uid:       str
    sub_cat:        str
    score:          float
    pixel_height:   float
    pixel_width:    float
    d_from_height:  Optional[float]
    d_from_width:   Optional[float]
    metric_depth:   float
    raw_da3_depth:  float
    anchor_weight:  float


def extract_vehicle_anchors(
    inst: dict,
    img_id: int,
    raw_depth: np.ndarray,
    intrinsics: CameraIntrinsics,
    image_hw: Tuple[int, int],
) -> Optional[VehicleAnchor]:
    sub_cat = inst["sub_cat"]
    prior   = VEHICLE_PRIORS.get(sub_cat)
    if prior is None:
        return None
    score = inst["score"]
    if score < MIN_MASK_SCORE:
        return None
    mask  = inst["mask"]
    H, W  = image_hw

    if mask.shape != (H, W):
        return None

    # ── Always derive tight bbox from mask pixels (ignores slack in manifest box) ──
    ys, xs = np.where(mask)
    if len(ys) == 0:
        return None
    x1 = max(0, int(xs.min()));  x2 = min(W, int(xs.max()))
    y1 = max(0, int(ys.min()));  y2 = min(H, int(ys.max()))
    ph, pw = float(y2 - y1), float(x2 - x1)

    if ph < MIN_PIXEL_DIM and pw < MIN_PIXEL_DIM:
        return None

    # ── Depth sample: 30th-percentile favours the near/dense vehicle body ────────
    # (avoids sky/background pixels that may leak into a loose mask boundary)
    mask_vals = raw_depth[mask]
    if mask_vals.size == 0:
        return None
    raw_med = float(np.median(mask_vals))

    # ── Orientation-aware dimension selection ────────────────────────────────────
    # Masks cover the FULL vehicle body.  When a car is side-on its pixel WIDTH
    # represents the car LENGTH (~4.5 m), not width (~1.8 m) — using it would
    # give a badly wrong depth.  Height is reliable at any orientation because
    # the roof-to-ground extent is stable regardless of yaw.
    #
    # aspect = pw/ph:  < 1.8 → frontal/rear-on (both dims usable)
    #                  1.8–2.5 → oblique (height only, down-weight width)
    #                  > 2.5 → side-on  (height only)
    aspect = pw / (ph + 1e-6)
    use_height = ph >= MIN_PIXEL_DIM
    if aspect < 1.8:
        use_width = pw >= MIN_PIXEL_DIM          # frontal/rear: width ≈ car width
        w_width_scale = 1.0
    elif aspect < 2.5:
        use_width = pw >= MIN_PIXEL_DIM          # oblique: allow but down-weight
        w_width_scale = 0.4
    else:
        use_width = False                        # side-on: width = car length, discard
        w_width_scale = 0.0

    depth_h_pair: Optional[Tuple[float, float]] = None
    depth_w_pair: Optional[Tuple[float, float]] = None

    if use_height and prior.real_height is not None:
        dh = intrinsics.fy * prior.real_height / ph
        if MIN_DEPTH < dh < MAX_DEPTH:
            depth_h_pair = (dh, prior.height_weight)

    if use_width and prior.real_width is not None:
        dw = intrinsics.fx * prior.real_width / pw
        if MIN_DEPTH < dw < MAX_DEPTH:
            depth_w_pair = (dw, prior.width_weight * w_width_scale)

    pairs = [p for p in [depth_h_pair, depth_w_pair] if p]
    if not pairs:
        return None

    depths  = np.array([p[0] for p in pairs])
    weights = np.array([p[1] for p in pairs])

    # Consistency check: if height/width estimates disagree >25%, keep the
    # higher-weight one only (tighter than old 1.30 threshold)
    if len(depths) == 2:
        ratio = max(depths) / (min(depths) + 1e-6)
        if ratio > 1.25:
            best    = int(np.argmax(weights))
            depths  = depths[best: best + 1]
            weights = weights[best: best + 1]

    metric_d  = float(np.average(depths, weights=weights))
    anchor_wt = prior.anchor_reliability * intrinsics.quality_weight * score * float(weights.sum())

    return VehicleAnchor(
        img_id=img_id, mask_uid=inst.get("uid", ""), sub_cat=sub_cat, score=score,
        pixel_height=ph, pixel_width=pw,
        d_from_height=depth_h_pair[0] if depth_h_pair else None,
        d_from_width=depth_w_pair[0]  if depth_w_pair  else None,
        metric_depth=metric_d, raw_da3_depth=raw_med, anchor_weight=anchor_wt,
    )

print("Vehicle anchor extraction defined.")

Vehicle anchor extraction defined.


## Cell 9 · Global Scale/Shift Estimation  (RANSAC + WLS)

Fits: $\text{scale} \cdot a_i + \text{shift} \approx d_i^{\mathrm{metric}}$

using RANSAC (500 iterations, 2-point minimal sampling) followed by a
weighted least-squares (WLS) refinement on all inliers ($|\text{residual}| < 2\,\mathrm{m}$).

In [11]:
@dataclass
class ScaleShiftResult:
    scale:     float
    shift:     float
    n_anchors: int
    n_inliers: int
    rmse_m:    float
    method:    str


def estimate_global_scale_shift(anchors: List[VehicleAnchor]) -> ScaleShiftResult:
    if len(anchors) < MIN_ANCHORS_FOR_FIT:
        log.warning("Only %d anchors — using scale=1, shift=0.", len(anchors))
        return ScaleShiftResult(scale=1.0, shift=0.0, n_anchors=len(anchors),
                                n_inliers=0, rmse_m=float("nan"), method="no_fit")

    raw  = np.array([a.raw_da3_depth  for a in anchors])
    met  = np.array([a.metric_depth   for a in anchors])
    wts  = np.array([a.anchor_weight  for a in anchors])
    wts /= (wts.sum() + 1e-12)

    best_scale, best_shift, best_inlier_count = 1.0, 0.0, 0
    rng = np.random.default_rng(42)

    for _ in range(RANSAC_ITERATIONS):
        idxs = rng.choice(len(anchors), size=2, replace=False)
        A    = np.vstack([raw[idxs], np.ones(2)]).T
        b    = met[idxs]
        try:
            params, *_ = np.linalg.lstsq(A, b, rcond=None)
        except np.linalg.LinAlgError:
            continue
        s, t = params
        if s <= 0:
            continue
        inlier_count = int((np.abs(s * raw + t - met) < RANSAC_INLIER_THR).sum())
        if inlier_count > best_inlier_count:
            best_inlier_count = inlier_count
            best_scale, best_shift = s, t

    # WLS refinement on inliers
    inlier_mask = np.abs(best_scale * raw + best_shift - met) < RANSAC_INLIER_THR
    if inlier_mask.sum() >= 2:
        ri, mi, wi = raw[inlier_mask], met[inlier_mask], wts[inlier_mask]
        wi /= wi.sum() + 1e-12
        W_mat = np.diag(wi)
        A_mat = np.vstack([ri, np.ones_like(ri)]).T
        try:
            p = np.linalg.lstsq(W_mat @ A_mat, W_mat @ mi, rcond=None)[0]
            if p[0] > 0:
                best_scale, best_shift = float(p[0]), float(p[1])
        except np.linalg.LinAlgError:
            pass

    method = "ransac_wls"
    if best_scale <= 0:
        log.warning("Non-positive scale — using median fallback.")
        best_scale = float(np.median(met) / (np.median(raw) + 1e-8))
        best_shift = 0.0
        method = "median_fallback"

    final_res   = np.abs(best_scale * raw + best_shift - met)
    inlier_mask = final_res < RANSAC_INLIER_THR
    rmse = float(np.sqrt(np.mean(final_res[inlier_mask] ** 2)) if inlier_mask.any() else float("nan"))

    log.info("Global scale=%.4f  shift=%.4f  inliers=%d/%d  RMSE=%.3fm  method=%s",
             best_scale, best_shift, int(inlier_mask.sum()), len(anchors), rmse, method)
    return ScaleShiftResult(scale=best_scale, shift=best_shift,
                            n_anchors=len(anchors), n_inliers=int(inlier_mask.sum()),
                            rmse_m=rmse, method=method)

print("Scale/shift estimation defined.")

Scale/shift estimation defined.


## Cell 10 · Evaluation Helpers  *(standard KITTI metrics)*

In [12]:
def compute_depth_errors(gt: np.ndarray, pred: np.ndarray, mask: np.ndarray) -> dict:
    gt_v    = gt[mask]
    pred_v  = np.maximum(pred[mask], 1e-8)
    thresh  = np.maximum(gt_v / pred_v, pred_v / gt_v)
    return dict(
        rmse     = float(np.sqrt(np.mean((gt_v - pred_v) ** 2))),
        abs_rel  = float(np.mean(np.abs(gt_v - pred_v) / gt_v)),
        sq_rel   = float(np.mean(((gt_v - pred_v) ** 2) / (gt_v ** 2))),
        rmse_log = float(np.sqrt(np.mean((np.log(gt_v) - np.log(pred_v)) ** 2))),
        delta1   = float(np.mean(thresh < 1.25)),
        delta2   = float(np.mean(thresh < 1.25 ** 2)),
        delta3   = float(np.mean(thresh < 1.25 ** 3)),
    )


def eigen_crop_mask(shape: Tuple[int, ...]) -> np.ndarray:
    h, w = shape[:2]
    mask = np.zeros((h, w), dtype=bool)
    mask[44: min(44 + 352, h), 0:w] = True
    return mask


def compute_all_scales(gt_depth: np.ndarray, pred_raw: np.ndarray,
                        va_depth: np.ndarray, eval_mask: np.ndarray) -> dict:
    gt_v   = gt_depth[eval_mask]
    pred_c = np.clip(pred_raw, MIN_DEPTH, MAX_DEPTH)
    results = {}

    # Unscaled
    m0 = compute_depth_errors(gt_depth, pred_c, eval_mask)
    results["unscaled"] = {"scale": 1.0, "shift": 0.0, "depth_map": pred_c, **m0,
                            "avg_depth": float(np.mean(pred_c[eval_mask])),
                            "min_depth": float(np.min(pred_c[eval_mask]))}

    # Median (oracle)
    s_med = float(np.median(gt_v) / np.median(pred_c[eval_mask]))
    p_med = np.clip(pred_c * s_med, MIN_DEPTH, MAX_DEPTH)
    m1 = compute_depth_errors(gt_depth, p_med, eval_mask)
    results["median"] = {"scale": s_med, "shift": 0.0, "depth_map": p_med, **m1,
                          "avg_depth": float(np.mean(p_med[eval_mask])),
                          "min_depth": float(np.min(p_med[eval_mask]))}

    # Mean (oracle)
    s_mean = float(np.mean(gt_v) / np.mean(pred_c[eval_mask]))
    p_mean = np.clip(pred_c * s_mean, MIN_DEPTH, MAX_DEPTH)
    m2 = compute_depth_errors(gt_depth, p_mean, eval_mask)
    results["mean"] = {"scale": s_mean, "shift": 0.0, "depth_map": p_mean, **m2,
                        "avg_depth": float(np.mean(p_mean[eval_mask])),
                        "min_depth": float(np.min(p_mean[eval_mask]))}

    # Affine (oracle OLS)
    pf = pred_c[eval_mask].ravel()
    A_mat = np.vstack([pf, np.ones_like(pf)]).T
    par, *_ = np.linalg.lstsq(A_mat, gt_v.ravel(), rcond=None)
    s_aff, t_aff = float(par[0]), float(par[1])
    p_aff = np.clip(s_aff * pred_c + t_aff, MIN_DEPTH, MAX_DEPTH)
    m3 = compute_depth_errors(gt_depth, p_aff, eval_mask)
    results["affine"] = {"scale": s_aff, "shift": t_aff, "depth_map": p_aff, **m3,
                          "avg_depth": float(np.mean(p_aff[eval_mask])),
                          "min_depth": float(np.min(p_aff[eval_mask]))}

    # Vehicle-anchored (ours)
    va_c = np.clip(va_depth, MIN_DEPTH, MAX_DEPTH)
    m4 = compute_depth_errors(gt_depth, va_c, eval_mask)
    results["va"] = {"scale": None, "shift": None, "depth_map": va_c, **m4,
                      "avg_depth": float(np.mean(va_c[eval_mask])),
                      "min_depth": float(np.min(va_c[eval_mask]))}

    return results

print("Evaluation helpers defined.")

Evaluation helpers defined.


## Cell 11 · CSV Column Schema

In [13]:
CSV_COLUMNS = [
    "img_id","file_name","mask_id","mask_uid","mask_score","super_cat","sub_cat","horizontal_angle",
    "pixel_height","pixel_width","d_from_height","d_from_width",
    "anchor_metric_depth","anchor_raw_depth","anchor_weight",
    "unscaled_rmse","unscaled_abs_rel","unscaled_sq_rel","unscaled_rmse_log",
    "unscaled_delta1","unscaled_delta2","unscaled_delta3","unscaled_avg_depth","unscaled_min_depth",
    "median_scale","median_rmse","median_abs_rel","median_sq_rel","median_rmse_log",
    "median_delta1","median_delta2","median_delta3","median_avg_depth","median_min_depth",
    "mean_scale","mean_rmse","mean_abs_rel","mean_sq_rel","mean_rmse_log",
    "mean_delta1","mean_delta2","mean_delta3","mean_avg_depth","mean_min_depth",
    "affine_scale","affine_shift","affine_rmse","affine_abs_rel","affine_sq_rel","affine_rmse_log",
    "affine_delta1","affine_delta2","affine_delta3","affine_avg_depth","affine_min_depth",
    "va_global_scale","va_global_shift","va_rmse","va_abs_rel","va_sq_rel","va_rmse_log",
    "va_delta1","va_delta2","va_delta3","va_avg_depth","va_min_depth",
    "lidar_avg_depth","lidar_min_depth",
    "rgb_image_path","gt_depth_map_path","da3_pred_map_path","mask_path",
]
print(f"CSV_COLUMNS defined ({len(CSV_COLUMNS)} columns).")

CSV_COLUMNS defined (72 columns).


## Cell 12 · Pass 1 — Vehicle Anchor Collection

Iterates over all frames, runs DA3 depth inference and collects vehicle anchors.
Intrinsics are predicted **once per sequence/drive** using DA3 and cached.

In [14]:
def run_pass1_collect_anchors(val_list: pd.DataFrame, da3_mdl) -> List[VehicleAnchor]:
    all_anchors: List[VehicleAnchor] = []
    anchor_rows: List[dict] = []

    # Pre-index image paths per drive for intrinsic prediction
    drive_to_paths: Dict[str, List[str]] = {}
    for row in val_list.itertuples():
        drive_root = os.path.join(TEST_ROOT, row.sequence, row.drive)
        if row.drive not in drive_to_paths:
            drive_to_paths[row.drive] = get_sequence_image_paths(drive_root)

    frames_no_masks = 0

    for index, row in tqdm(val_list.iterrows(), total=len(val_list),
                            desc="Pass 1 — Anchor Collection"):
        if index < START_INDEX:
            continue
        try:
            image_np = np.array(Image.open(row["rgb_path"]).convert("RGB"))
            H, W, _  = image_np.shape
            mask_dir = get_mask_dir(row["rgb_path"])
            vehicle_insts = load_masks_for_frame(mask_dir, super_cat_filter="vehicle")
            if not vehicle_insts:
                frames_no_masks += 1
                continue
            seq_paths = drive_to_paths.get(row["drive"], [])
            cam       = get_sequence_intrinsics(row["drive"], seq_paths, (W, H))
            raw_depth = run_da3_depth(da3_mdl, image_np)
            for inst in vehicle_insts:
                anchor = extract_vehicle_anchors(inst, int(index), raw_depth, cam, (H, W))
                if anchor is None:
                    continue
                all_anchors.append(anchor)
                anchor_rows.append({
                    "img_id": index, "drive": row["drive"], "filename": row["filename"],
                    "mask_uid": anchor.mask_uid, "sub_cat": anchor.sub_cat,
                    "score": round(anchor.score, 4),
                    "pixel_height": round(anchor.pixel_height, 2),
                    "pixel_width":  round(anchor.pixel_width,  2),
                    "d_from_height": round(anchor.d_from_height, 4) if anchor.d_from_height else None,
                    "d_from_width":  round(anchor.d_from_width,  4) if anchor.d_from_width  else None,
                    "metric_depth":  round(anchor.metric_depth,  4),
                    "raw_da3_depth": round(anchor.raw_da3_depth, 4),
                    "anchor_weight": round(anchor.anchor_weight, 6),
                    "fx": round(cam.fx, 2), "fy": round(cam.fy, 2),
                    "intrin_source": cam.source,
                    "intrin_quality": round(cam.quality_weight, 4),
                })
        except Exception as exc:
            log.error("Pass 1 error at index %d: %s", index, exc, exc_info=True)

    log.info("Pass 1 complete: %d anchors  (%d frames had no vehicle masks).",
             len(all_anchors), frames_no_masks)

    if anchor_rows:
        pd.DataFrame(anchor_rows).to_csv(ANCHORS_CSV, index=False)
        log.info("Anchors CSV saved → %s", ANCHORS_CSV)

    if _sequence_intrin_cache:
        pd.DataFrame([{"drive": d, **c.to_dict()}
                       for d, c in _sequence_intrin_cache.items()]
        ).to_csv(INTRINSICS_CSV, index=False)
        log.info("Intrinsics CSV saved → %s", INTRINSICS_CSV)

    return all_anchors

print("run_pass1_collect_anchors() defined.")

run_pass1_collect_anchors() defined.


## Cell 13 · Pass 2 — Evaluation

Applies the global vehicle-anchored (scale, shift) to raw DA3 depth, then
computes all five scale variants vs LiDAR GT for every mask in every frame.

In [15]:
def run_pass2_evaluate(val_list: pd.DataFrame, da3_mdl,
                        va_scale: float, va_shift: float) -> None:
    RGB_DIR  = os.path.join(MAIN_OUTPUT_DIR, "rgb_images");  os.makedirs(RGB_DIR, exist_ok=True)
    GT_DIR   = os.path.join(MAIN_OUTPUT_DIR, "gt_depth");    os.makedirs(GT_DIR,  exist_ok=True)
    DA3_DIR  = os.path.join(MAIN_OUTPUT_DIR, "da3_pred");    os.makedirs(DA3_DIR, exist_ok=True)
    MASK_OUT = os.path.join(MAIN_OUTPUT_DIR, "masks");       os.makedirs(MASK_OUT,exist_ok=True)

    if not os.path.exists(OUTPUT_CSV_PATH):
        pd.DataFrame(columns=CSV_COLUMNS).to_csv(OUTPUT_CSV_PATH, index=False)

    frames_no_masks = 0

    for index, row in tqdm(val_list.iterrows(), total=len(val_list), desc="Pass 2 — Evaluation"):
        if index < START_INDEX:
            continue
        base_fn = row["folder"] + "_" + row["drive"] + "_" + os.path.splitext(row["filename"])[0]
        try:
            image_np = np.array(Image.open(row["rgb_path"]).convert("RGB"))
            gt_depth = np.array(Image.open(row["gt_path"])).astype(np.uint16) / 256.0
            H, W, _  = image_np.shape
            cx_img   = W / 2.0

            mask_dir      = get_mask_dir(row["rgb_path"])
            all_instances = load_masks_for_frame(mask_dir)
            if not all_instances:
                frames_no_masks += 1
                continue

            pred_raw = run_da3_depth(da3_mdl, image_np)
            va_depth = np.clip(va_scale * pred_raw + va_shift, MIN_DEPTH, MAX_DEPTH)

            rgb_out = os.path.join(RGB_DIR, f"{base_fn}.png")
            gt_out  = os.path.join(GT_DIR,  f"{base_fn}.npy")
            da3_out = os.path.join(DA3_DIR, f"{base_fn}.npy")
            Image.fromarray(image_np).save(rgb_out)
            if SAVE_NPY:
                np.save(gt_out, gt_depth); np.save(da3_out, pred_raw)

            crop_mask = eigen_crop_mask(gt_depth.shape)

            for inst in all_instances:
                seg_mask  = inst["mask"]; score = inst["score"]
                super_cat = inst["super_cat"]; sub_cat = inst["sub_cat"]
                mask_idx  = inst["local_idx"]; mask_uid = inst.get("uid", "")

                if score < MIN_MASK_SCORE:
                    continue
                if seg_mask.shape != (H, W):
                    continue

                eval_mask = seg_mask & (gt_depth > MIN_DEPTH) & (gt_depth < MAX_DEPTH) & crop_mask
                if not np.any(eval_mask):
                    continue

                # Anchor diagnostics (vehicle only)
                anch_h = anch_w = anch_dh = anch_dw = anch_met = anch_raw = anch_wt = None
                if super_cat == "vehicle":
                    prior = VEHICLE_PRIORS.get(sub_cat)
                    if prior:
                        box = inst.get("box")
                        if box:
                            x1,y1,x2,y2 = (int(v) for v in box)
                        else:
                            ys2,xs2 = np.where(seg_mask)
                            x1,y1,x2,y2 = (int(xs2.min()),int(ys2.min()),int(xs2.max()),int(ys2.max())) if len(ys2) else (0,0,0,0)
                        anch_h = float(y2-y1); anch_w = float(x2-x1)
                        cam = _sequence_intrin_cache.get(row["drive"])
                        if cam:
                            if prior.real_height and anch_h >= MIN_PIXEL_DIM:
                                anch_dh = round(cam.fy * prior.real_height / anch_h, 4)
                            if prior.real_width and anch_w >= MIN_PIXEL_DIM:
                                anch_dw = round(cam.fx * prior.real_width  / anch_w, 4)
                            vals = [v for v in [anch_dh,anch_dw] if v and MIN_DEPTH < v < MAX_DEPTH]
                            anch_met = round(float(np.mean(vals)), 4) if vals else None
                        mv = pred_raw[seg_mask]
                        if mv.size > 0:
                            anch_raw = round(float(np.median(mv)), 4)
                        # Compute anchor weight (mirrors extract_vehicle_anchors logic)
                        wts_list = []
                        if anch_dh is not None and MIN_DEPTH < anch_dh < MAX_DEPTH:
                            wts_list.append(prior.height_weight)
                        if anch_dw is not None and MIN_DEPTH < anch_dw < MAX_DEPTH:
                            wts_list.append(prior.width_weight)
                        if wts_list and cam:
                            anch_wt = round(
                                prior.anchor_reliability * cam.quality_weight * inst["score"] * sum(wts_list), 6
                            )

                scale_res  = compute_all_scales(gt_depth, pred_raw, va_depth, eval_mask)
                lidar_vals = gt_depth[eval_mask]
                ys_seg, xs_seg = np.where(seg_mask)
                cx2 = float(np.mean(xs_seg)) if len(xs_seg) else cx_img
                angle_deg = math.degrees(math.atan((cx2 - cx_img) / FOCAL_LENGTH_X))

                mask_path_out = str(inst.get("path",""))
                if SAVE_NPY and mask_path_out:
                    np.save(os.path.join(MASK_OUT, f"{base_fn}_mask_{mask_idx}.npy"), eval_mask)

                def _m(v,k): return scale_res[v].get(k)
                csv_row = {
                    "img_id":index,"file_name":base_fn,"mask_id":mask_idx,"mask_uid":mask_uid,
                    "mask_score":round(score,4),"super_cat":super_cat,"sub_cat":sub_cat,
                    "horizontal_angle":round(angle_deg,4),
                    "pixel_height":anch_h,"pixel_width":anch_w,"d_from_height":anch_dh,
                    "d_from_width":anch_dw,"anchor_metric_depth":anch_met,"anchor_raw_depth":anch_raw,
                    "anchor_weight":anch_wt,
                    "unscaled_rmse":_m("unscaled","rmse"),"unscaled_abs_rel":_m("unscaled","abs_rel"),
                    "unscaled_sq_rel":_m("unscaled","sq_rel"),"unscaled_rmse_log":_m("unscaled","rmse_log"),
                    "unscaled_delta1":_m("unscaled","delta1"),"unscaled_delta2":_m("unscaled","delta2"),
                    "unscaled_delta3":_m("unscaled","delta3"),"unscaled_avg_depth":_m("unscaled","avg_depth"),
                    "unscaled_min_depth":_m("unscaled","min_depth"),
                    "median_scale":_m("median","scale"),"median_rmse":_m("median","rmse"),
                    "median_abs_rel":_m("median","abs_rel"),"median_sq_rel":_m("median","sq_rel"),
                    "median_rmse_log":_m("median","rmse_log"),"median_delta1":_m("median","delta1"),
                    "median_delta2":_m("median","delta2"),"median_delta3":_m("median","delta3"),
                    "median_avg_depth":_m("median","avg_depth"),"median_min_depth":_m("median","min_depth"),
                    "mean_scale":_m("mean","scale"),"mean_rmse":_m("mean","rmse"),
                    "mean_abs_rel":_m("mean","abs_rel"),"mean_sq_rel":_m("mean","sq_rel"),
                    "mean_rmse_log":_m("mean","rmse_log"),"mean_delta1":_m("mean","delta1"),
                    "mean_delta2":_m("mean","delta2"),"mean_delta3":_m("mean","delta3"),
                    "mean_avg_depth":_m("mean","avg_depth"),"mean_min_depth":_m("mean","min_depth"),
                    "affine_scale":_m("affine","scale"),"affine_shift":_m("affine","shift"),
                    "affine_rmse":_m("affine","rmse"),"affine_abs_rel":_m("affine","abs_rel"),
                    "affine_sq_rel":_m("affine","sq_rel"),"affine_rmse_log":_m("affine","rmse_log"),
                    "affine_delta1":_m("affine","delta1"),"affine_delta2":_m("affine","delta2"),
                    "affine_delta3":_m("affine","delta3"),"affine_avg_depth":_m("affine","avg_depth"),
                    "affine_min_depth":_m("affine","min_depth"),
                    "va_global_scale":round(va_scale,6),"va_global_shift":round(va_shift,6),
                    "va_rmse":_m("va","rmse"),"va_abs_rel":_m("va","abs_rel"),
                    "va_sq_rel":_m("va","sq_rel"),"va_rmse_log":_m("va","rmse_log"),
                    "va_delta1":_m("va","delta1"),"va_delta2":_m("va","delta2"),
                    "va_delta3":_m("va","delta3"),"va_avg_depth":_m("va","avg_depth"),
                    "va_min_depth":_m("va","min_depth"),
                    "lidar_avg_depth":round(float(np.mean(lidar_vals)),4),
                    "lidar_min_depth":round(float(np.min(lidar_vals)),4),
                    "rgb_image_path":rgb_out,"gt_depth_map_path":gt_out,
                    "da3_pred_map_path":da3_out,"mask_path":mask_path_out,
                }
                pd.DataFrame([csv_row]).to_csv(OUTPUT_CSV_PATH, mode="a", header=False, index=False)

        except Exception as exc:
            log.error("Pass 2 error at index %d (%s): %s", index, base_fn, exc, exc_info=True)

    log.info("Pass 2 complete → %s", OUTPUT_CSV_PATH)

print("run_pass2_evaluate() defined.")

run_pass2_evaluate() defined.


## Cell 14 · Analysis & Visualisation Helpers

In [16]:
SCALE_COLORS = {
    "unscaled": "#e74c3c",
    "median":   "#27ae60",
    "mean":     "#2980b9",
    "affine":   "#8e44ad",
    "va":       "#e67e22",
}
SCALE_LABELS = {
    "unscaled": "Unscaled",
    "median":   "Median Scale (oracle)",
    "mean":     "Mean Scale (oracle)",
    "affine":   "Affine Scale (oracle)",
    "va":       "Vehicle-Anchored (ours)",
}
ALL_KEYS = list(SCALE_COLORS.keys())


def _save_fig(fig: plt.Figure, name: str, dpi: int = 150) -> None:
    os.makedirs(PLOT_DIR, exist_ok=True)
    path = os.path.join(PLOT_DIR, name)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    log.info("  ✓  %s", name)


def load_results() -> pd.DataFrame:
    df = pd.read_csv(OUTPUT_CSV_PATH)
    df = df.dropna(subset=["unscaled_rmse","median_rmse","mean_rmse",
                             "affine_rmse","va_rmse","lidar_avg_depth"])
    return df

print("Analysis colour / label config and helpers defined.")

Analysis colour / label config and helpers defined.


## Cell 15 · Plot Functions  *(13 plots)*

In [17]:
def plot_gt_vs_pred(df: pd.DataFrame) -> None:
    variants = [
        ("Unscaled",              "lidar_avg_depth", "unscaled_avg_depth"),
        ("Median Scale (oracle)", "lidar_avg_depth", "median_avg_depth"),
        ("Affine Scale (oracle)", "lidar_avg_depth", "affine_avg_depth"),
        ("Vehicle-Anchored",      "lidar_avg_depth", "va_avg_depth"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    axes = axes.flatten()
    colors = [SCALE_COLORS["unscaled"], SCALE_COLORS["median"],
              SCALE_COLORS["affine"],   SCALE_COLORS["va"]]
    for ax, (title, xcol, ycol), color in zip(axes, variants, colors):
        sns.regplot(ax=ax, data=df, x=xcol, y=ycol,
                    scatter_kws={"s": 25, "alpha": 0.35, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.plot([0, MAX_DEPTH], [0, MAX_DEPTH], "k--", linewidth=1.5, label="Ideal 1:1")
        ax.set_title(f"DA3METRIC-LARGE  |  {title}", fontsize=11)
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Predicted Depth (m)")
        ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, MAX_DEPTH); ax.legend(fontsize=9)
    fig.suptitle("Ground-Truth vs Predicted Depth — Key Scale Variants", fontsize=14, y=1.01)
    plt.tight_layout(); _save_fig(fig, "01_gt_vs_pred_all_scales.png")


# ── Plot 02: GT Depth vs RMSE ─────────────────────────────────────────────────
def plot_gt_vs_rmse(df: pd.DataFrame) -> None:
    variants = [(SCALE_LABELS[k], f"{k}_rmse", SCALE_COLORS[k]) for k in ALL_KEYS]
    ymax = df[[f"{k}_rmse" for k in ALL_KEYS]].max().max() * 1.12
    fig, axes = plt.subplots(2, 3, figsize=(21, 12)); axes = axes.flatten()
    for ax, (title, ycol, color) in zip(axes, variants):
        sns.regplot(ax=ax, data=df, x="lidar_avg_depth", y=ycol,
                    scatter_kws={"s": 25, "alpha": 0.35, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.set_title(f"{title}", fontsize=11)
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("RMSE (m)")
        ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, ymax)
    for ax in axes[len(variants):]:
        ax.set_visible(False)
    fig.suptitle("GT Depth vs RMSE — All Scale Variants", fontsize=14, y=1.01)
    plt.tight_layout(); _save_fig(fig, "02_gt_vs_rmse_all_scales.png")


# ── Plot 03: Horizontal Angle vs RMSE ────────────────────────────────────────
def plot_angle_vs_rmse(df: pd.DataFrame) -> None:
    variants = [(SCALE_LABELS[k], f"{k}_rmse", SCALE_COLORS[k]) for k in ALL_KEYS]
    ymax_raw = df[[f"{k}_rmse" for k in ALL_KEYS]].max().max()
    ymax = ymax_raw * 1.12 if np.isfinite(ymax_raw) and ymax_raw > 0 else 1.0
    fig, axes = plt.subplots(2, 3, figsize=(21, 12)); axes = axes.flatten()
    for ax, (title, ycol, color) in zip(axes, variants):
        sns.regplot(ax=ax, data=df, x="horizontal_angle", y=ycol,
                    scatter_kws={"s": 25, "alpha": 0.35, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.set_title(f"{title}: Angle vs RMSE", fontsize=11)
        ax.set_xlabel("Horizontal Angle (°)"); ax.set_ylabel("RMSE (m)"); ax.set_ylim(0, ymax)
    for ax in axes[len(variants):]:
        ax.set_visible(False)
    fig.suptitle("Horizontal Angle vs RMSE — All Scale Variants", fontsize=14, y=1.01)
    plt.tight_layout(); _save_fig(fig, "03_angle_vs_rmse_all_scales.png")


# ── Plot 04: Scale Method Comparison Bar Chart ────────────────────────────────
def plot_scale_comparison_bars(df: pd.DataFrame) -> None:
    metrics_spec = [
        ("RMSE (m)  ↓", [f"{k}_rmse"    for k in ALL_KEYS]),
        ("Abs Rel   ↓", [f"{k}_abs_rel"  for k in ALL_KEYS]),
        ("δ < 1.25  ↑", [f"{k}_delta1"  for k in ALL_KEYS]),
    ]
    labels = [SCALE_LABELS[k] for k in ALL_KEYS]
    colors = [SCALE_COLORS[k] for k in ALL_KEYS]
    fig, axes = plt.subplots(1, 3, figsize=(21, 6))
    for ax, (metric_name, cols) in zip(axes, metrics_spec):
        vals = [df[c].mean() for c in cols]
        bars = ax.bar(labels, vals, color=colors, edgecolor="white", width=0.6)
        ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=9)
        ax.set_title(metric_name, fontsize=12); ax.set_ylabel(metric_name)
        ax.set_ylim(0, max(vals) * 1.25); ax.grid(axis="y", alpha=0.35)
        ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=9)
    fig.suptitle("DA3METRIC-LARGE — Scale Method Comparison", fontsize=14, y=1.02)
    plt.tight_layout(); _save_fig(fig, "04_scale_comparison_bars.png")


# ── Plot 05: Predicted vs LiDAR by Object Category ───────────────────────────
def plot_model_vs_lidar_by_category(df: pd.DataFrame) -> None:
    cats = sorted(df["super_cat"].dropna().unique())
    n = max(len(cats), 1)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 6))
    if n == 1:
        axes = [axes]
    for ax, cat in zip(axes, cats):
        sub = df[df["super_cat"] == cat]
        for key in ALL_KEYS:
            ax.scatter(sub["lidar_avg_depth"], sub[f"{key}_avg_depth"],
                       s=18, alpha=0.4, c=SCALE_COLORS[key], label=SCALE_LABELS[key])
        ax.plot([0, MAX_DEPTH], [0, MAX_DEPTH], "k--", linewidth=1.5, label="Ideal 1:1")
        ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, MAX_DEPTH)
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Predicted Depth (m)")
        ax.set_title(f"Category: {cat}", fontsize=12); ax.legend(fontsize=7)
    fig.suptitle("Predicted Depth vs LiDAR GT — by Object Category", fontsize=13)
    plt.tight_layout(); _save_fig(fig, "05_model_vs_lidar_by_category.png")


# ── Plot 06: RMSE Distribution Histograms ─────────────────────────────────────
def plot_rmse_distributions(df: pd.DataFrame) -> None:
    fig, axes = plt.subplots(2, 3, figsize=(21, 12)); axes = axes.flatten()
    for ax, key in zip(axes, ALL_KEYS):
        col  = f"{key}_rmse"
        vals = df[col].dropna()
        ax.hist(vals, bins=60, color=SCALE_COLORS[key], alpha=0.72, edgecolor="white")
        ax.axvline(vals.mean(),   color="black", linestyle="--", linewidth=1.8,
                   label=f"Mean   = {vals.mean():.3f} m")
        ax.axvline(vals.median(), color="grey",  linestyle=":",  linewidth=1.8,
                   label=f"Median = {vals.median():.3f} m")
        ax.set_title(f"RMSE Distribution — {SCALE_LABELS[key]}", fontsize=11)
        ax.set_xlabel("RMSE (m)"); ax.set_ylabel("Count"); ax.legend(fontsize=9)
    for ax in axes[len(ALL_KEYS):]:
        ax.set_visible(False)
    fig.suptitle("RMSE Distributions Across All Masks", fontsize=13)
    plt.tight_layout(); _save_fig(fig, "06_rmse_distributions.png")


# ── Plot 07: Scale Factor Distributions ──────────────────────────────────────
def plot_scale_distributions(df: pd.DataFrame) -> None:
    spec = [
        ("median_scale", "Median Scale (oracle)", SCALE_COLORS["median"]),
        ("mean_scale",   "Mean Scale (oracle)",   SCALE_COLORS["mean"]),
        ("affine_scale", "Affine Scale (oracle)", SCALE_COLORS["affine"]),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (col, label, color) in zip(axes, spec):
        vals = df[col].dropna()
        ax.hist(vals, bins=60, color=color, alpha=0.72, edgecolor="white")
        va_s = df["va_global_scale"].iloc[0] if "va_global_scale" in df else None
        ax.axvline(1.0, color="black", linestyle="--", linewidth=1.8, label="Scale = 1.0")
        ax.axvline(vals.mean(), color="red", linestyle=":", linewidth=1.8,
                   label=f"Dataset Mean = {vals.mean():.3f}")
        if va_s is not None:
            ax.axvline(va_s, color=SCALE_COLORS["va"], linestyle="-.",
                       linewidth=2.0, label=f"VA Scale = {va_s:.3f}")
        ax.set_title(f"{label} Distribution", fontsize=11)
        ax.set_xlabel("Scale Factor"); ax.set_ylabel("Count"); ax.legend(fontsize=9)
    fig.suptitle("Scale Factor Distributions", fontsize=13)
    plt.tight_layout(); _save_fig(fig, "07_scale_distributions.png")


# ── Plot 08: 3D Error Surfaces ────────────────────────────────────────────────
def plot_3d_error_surfaces(df: pd.DataFrame, degree: int = 2) -> None:
    keys    = ["unscaled", "affine", "va"]
    colors  = [SCALE_COLORS[k] for k in keys]
    labels  = [SCALE_LABELS[k] for k in keys]
    xcol, ycol = "lidar_avg_depth", "horizontal_angle"
    df_c = df.dropna(subset=[xcol, ycol] + [f"{k}_rmse" for k in keys])
    if df_c.empty:
        return
    x_min, x_max = df_c[xcol].min(), df_c[xcol].max()
    y_min, y_max = df_c[ycol].min(), df_c[ycol].max()
    z_vals = [df_c[f"{k}_rmse"].values for k in keys]
    z_min  = min(z.min() for z in z_vals)
    z_max  = max(z.max() for z in z_vals)
    buf = lambda r: r * 0.08
    fig = plt.figure(figsize=(22, 6))
    fig.suptitle(f"3D Error Analysis (poly deg={degree}) — key variants", fontsize=13)
    for sp_idx, (key, color, label) in enumerate(zip(keys, colors, labels), start=1):
        ax = fig.add_subplot(1, 3, sp_idx, projection="3d")
        zcol = f"{key}_rmse"
        ax.scatter(df_c[xcol], df_c[ycol], df_c[zcol], c=color, alpha=0.35, s=8)
        Xf = df_c[[xcol, ycol]].values; Zf = df_c[zcol].values
        pipe = make_pipeline(PolynomialFeatures(degree), LinearRegression())
        pipe.fit(Xf, Zf)
        xs, ys = np.meshgrid(np.linspace(x_min, x_max, 40),
                              np.linspace(y_min, y_max, 40))
        zs = pipe.predict(np.c_[xs.ravel(), ys.ravel()]).reshape(xs.shape)
        ax.plot_surface(xs, ys, zs, color=color, alpha=0.22, rstride=1, cstride=1)
        ax.set_xlabel("GT Depth (m)", fontsize=8); ax.set_ylabel("H. Angle (°)", fontsize=8)
        ax.set_zlabel("RMSE (m)", fontsize=8); ax.set_title(label, fontsize=10)
        bx, by, bz = buf(x_max-x_min), buf(y_max-y_min), buf(z_max-z_min)
        ax.set_xlim(x_min-bx, x_max+bx); ax.set_ylim(y_min-by, y_max+by)
        ax.set_zlim(z_min-bz, z_max+bz)
    plt.tight_layout(); _save_fig(fig, "08_3d_error_surfaces.png")


# ── Plot 09: Depth-Bin Bias Analysis ─────────────────────────────────────────
def plot_depth_bin_analysis(df: pd.DataFrame) -> None:
    bins   = np.arange(0, MAX_DEPTH + 10, 10)
    labels = [f"{int(b)}–{int(b+10)}m" for b in bins[:-1]]
    df2    = df.copy()
    df2["depth_bin"] = pd.cut(df2["lidar_avg_depth"], bins=bins, labels=labels)
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    ax = axes[0]
    for key in ALL_KEYS:
        agg = df2.groupby("depth_bin", observed=True)[f"{key}_rmse"].mean()
        ax.plot(agg.index, agg.values, "o-",
                color=SCALE_COLORS[key], label=SCALE_LABELS[key], linewidth=2)
    ax.set_xlabel("Depth Bin"); ax.set_ylabel("Mean RMSE (m)")
    ax.set_title("RMSE by Depth Bin")
    ax.legend(fontsize=8); ax.tick_params(axis="x", rotation=45); ax.grid(alpha=0.3)
    ax = axes[1]
    for key in ALL_KEYS:
        agg_pred  = df2.groupby("depth_bin", observed=True)[f"{key}_avg_depth"].mean()
        agg_lidar = df2.groupby("depth_bin", observed=True)["lidar_avg_depth"].mean()
        bias = agg_pred - agg_lidar
        ax.plot(bias.index, bias.values, "o-",
                color=SCALE_COLORS[key], label=SCALE_LABELS[key], linewidth=2)
    ax.axhline(0, color="black", linestyle="--", linewidth=1.5, label="Zero bias")
    ax.set_xlabel("Depth Bin"); ax.set_ylabel("Prediction Bias (m)")
    ax.set_title("Depth Bias by Bin (Pred − LiDAR)")
    ax.legend(fontsize=8); ax.tick_params(axis="x", rotation=45); ax.grid(alpha=0.3)
    fig.suptitle("Depth-Bin Analysis — DA3METRIC-LARGE", fontsize=13)
    plt.tight_layout(); _save_fig(fig, "09_depth_bin_analysis.png")


# ── Plot 10: RMSE per Object Category ────────────────────────────────────────
def plot_rmse_by_category(df: pd.DataFrame) -> None:
    cats = sorted(df["super_cat"].dropna().unique())
    if len(cats) < 1:
        return
    n_keys = len(ALL_KEYS)
    x = np.arange(len(cats))
    w = 0.15
    fig, ax = plt.subplots(figsize=(14, 6))
    for i, key in enumerate(ALL_KEYS):
        vals   = [df[df["super_cat"] == c][f"{key}_rmse"].mean() for c in cats]
        offset = (i - (n_keys - 1) / 2.0) * w
        bars   = ax.bar(x + offset, vals, width=w,
                        label=SCALE_LABELS[key], color=SCALE_COLORS[key])
        ax.bar_label(bars, fmt="%.2f", fontsize=7, padding=2)
    ax.set_xticks(x); ax.set_xticklabels(cats, fontsize=11)
    ax.set_ylabel("Mean RMSE (m)")
    ax.set_title("RMSE per Object Category — DA3METRIC-LARGE + Vehicle-Anchored", fontsize=12)
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.35)
    plt.tight_layout(); _save_fig(fig, "10_rmse_by_category.png")


# ── Plot 11: Scale Factor vs Depth ───────────────────────────────────────────
def plot_scale_vs_depth(df: pd.DataFrame) -> None:
    spec = [
        ("median_scale", "Median Scale (oracle)", SCALE_COLORS["median"]),
        ("mean_scale",   "Mean Scale (oracle)",   SCALE_COLORS["mean"]),
        ("affine_scale", "Affine Scale (oracle)", SCALE_COLORS["affine"]),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (col, label, color) in zip(axes, spec):
        sns.regplot(ax=ax, data=df, x="lidar_avg_depth", y=col,
                    scatter_kws={"s": 18, "alpha": 0.30, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.axhline(1.0, color="black", linestyle="--", linewidth=1.5, label="Scale = 1.0")
        if "va_global_scale" in df:
            va_s = df["va_global_scale"].iloc[0]
            ax.axhline(va_s, color=SCALE_COLORS["va"], linestyle="-.",
                       linewidth=2.0, label=f"VA Scale = {va_s:.3f}")
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Scale Factor")
        ax.set_title(f"{label} vs Depth"); ax.legend(fontsize=9)
    fig.suptitle("Scale Factor vs Ground-Truth Depth", fontsize=13)
    plt.tight_layout(); _save_fig(fig, "11_scale_vs_depth.png")


# ── Plot 12: Vehicle Anchor Quality ──────────────────────────────────────────
def plot_anchor_quality(df: pd.DataFrame) -> None:
    """Show depth-anchor accuracy and sub-category breakdown for vehicle masks."""
    veh = df[df["super_cat"] == "vehicle"].dropna(
        subset=["anchor_metric_depth", "anchor_raw_depth"]
    )
    if veh.empty:
        log.warning("No vehicle rows with anchor data — skipping Plot 12.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(21, 6))

    # (a) Anchor metric depth vs LiDAR
    ax = axes[0]
    sub_cats = veh["sub_cat"].unique()
    for sc in sub_cats:
        s = veh[veh["sub_cat"] == sc]
        ax.scatter(s["lidar_avg_depth"], s["anchor_metric_depth"],
                   s=20, alpha=0.5, label=sc)
    ax.plot([0, MAX_DEPTH], [0, MAX_DEPTH], "k--", linewidth=1.5, label="Ideal 1:1")
    ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Anchor Metric Depth (m)")
    ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, MAX_DEPTH)
    ax.set_title("Vehicle Anchor Accuracy"); ax.legend(fontsize=9)

    # (b) Anchor error histogram
    ax = axes[1]
    anchor_err = veh["anchor_metric_depth"] - veh["lidar_avg_depth"]
    ax.hist(anchor_err.dropna(), bins=60, color=SCALE_COLORS["va"],
            alpha=0.75, edgecolor="white")
    ax.axvline(anchor_err.mean(), color="black", linestyle="--",
               label=f"Mean error = {anchor_err.mean():.2f} m")
    ax.set_xlabel("Anchor Error (m)"); ax.set_ylabel("Count")
    ax.set_title("Vehicle Anchor Depth Error Distribution"); ax.legend()

    # (c) d_from_height vs d_from_width scatter
    ax = axes[2]
    both = veh.dropna(subset=["d_from_height", "d_from_width"])
    if not both.empty:
        ax.scatter(both["d_from_height"], both["d_from_width"],
                   s=18, alpha=0.35, color=SCALE_COLORS["va"])
        lim_max = max(both[["d_from_height","d_from_width"]].max().max(), 1)
        ax.plot([0, lim_max], [0, lim_max], "k--", linewidth=1.5, label="Ideal 1:1")
        ax.set_xlabel("Depth from Height Anchor (m)")
        ax.set_ylabel("Depth from Width Anchor (m)")
        ax.set_title("Height-Anchor vs Width-Anchor Consistency"); ax.legend()
    else:
        ax.set_visible(False)

    fig.suptitle("Vehicle Anchor Quality Analysis", fontsize=13)
    plt.tight_layout(); _save_fig(fig, "12_vehicle_anchor_quality.png")


# ── Plot 00: Summary Table ────────────────────────────────────────────────────
def plot_summary_table(df: pd.DataFrame) -> None:
    va_s = df["va_global_scale"].mean() if "va_global_scale" in df else float("nan")
    va_t = df["va_global_shift"].mean() if "va_global_shift" in df else float("nan")
    rows_data, specs = [], [
        ("Unscaled",               "unscaled_rmse","unscaled_abs_rel","unscaled_delta1",None,None,False),
        ("Median Scale (oracle)",  "median_rmse",  "median_abs_rel",  "median_delta1",  "median_scale",None,False),
        ("Mean Scale (oracle)",    "mean_rmse",    "mean_abs_rel",    "mean_delta1",    "mean_scale",  None,False),
        ("Affine Scale (oracle)",  "affine_rmse",  "affine_abs_rel",  "affine_delta1",  "affine_scale","affine_shift",False),
        ("Vehicle-Anchored (ours)","va_rmse",      "va_abs_rel",      "va_delta1",      None,None,True),
    ]
    for label, rc, ac, dc, sc, shc, is_va in specs:
        sv  = df[sc].mean()  if sc   else (va_s if is_va else 1.0)
        shv = df[shc].mean() if shc  else (va_t if is_va else 0.0)
        rows_data.append([
            label,
            f"{sv:.4f}"  if not is_va else f"{va_s:.4f}",
            f"{shv:.4f}" if (shc or is_va) else "—",
            f"{df[rc].mean():.4f}",
            f"{df[ac].mean():.4f}",
            f"{df[dc].mean():.4f}",
        ])
    fig, ax = plt.subplots(figsize=(16, 3)); ax.axis("off")
    tbl = ax.table(
        cellText=rows_data,
        colLabels=["Scale Method", "Scale", "Shift (m)",
                   "Avg RMSE (m)", "Avg Abs Rel", "Avg δ<1.25"],
        cellLoc="center", loc="center", bbox=[0, 0, 1, 1],
    )
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 2.0)
    for j in range(6):
        tbl[0, j].set_facecolor("#2c3e50")
        tbl[0, j].set_text_props(color="white", fontweight="bold")
    for i in range(1, len(rows_data) + 1):
        bg = "#fff7e6" if i == len(rows_data) else ("white" if i % 2 == 0 else "#f0f4f8")
        for j in range(6):
            tbl[i, j].set_facecolor(bg)
    # Highlight VA row in orange
    for j in range(6):
        tbl[len(rows_data), j].set_edgecolor(SCALE_COLORS["va"])
        tbl[len(rows_data), j].set_linewidth(2)
    fig.suptitle("DA3METRIC-LARGE — Scale Method Summary (Vehicle-Anchored vs Oracle)",
                 fontsize=13, fontweight="bold", y=1.05)
    plt.tight_layout(); _save_fig(fig, "00_summary_table.png", dpi=180)


def generate_all_plots(df: pd.DataFrame) -> None:
    """Generate and save all 13 analysis plots."""
    if df.empty:
        log.warning("DataFrame is empty — skipping plot generation.")
        return
    sns.set_style("whitegrid"); sns.set_context("paper", font_scale=1.1)
    log.info("Generating analysis plots → %s", PLOT_DIR)
    plot_summary_table(df)
    plot_gt_vs_pred(df);           plot_gt_vs_rmse(df)
    plot_angle_vs_rmse(df);        plot_scale_comparison_bars(df)
    plot_model_vs_lidar_by_category(df)
    plot_rmse_distributions(df);   plot_scale_distributions(df)
    plot_3d_error_surfaces(df);    plot_depth_bin_analysis(df)
    plot_rmse_by_category(df);     plot_scale_vs_depth(df)
    plot_anchor_quality(df)
    log.info("All 13 plots saved to %s", PLOT_DIR)

# ══════════════════════════════════════════════════════════════════════════════
# 14.  TERMINAL SUMMARY PRINTER
# ══════════════════════════════════════════════════════════════════════════════

def print_summary(df: pd.DataFrame, ss_result: Optional[ScaleShiftResult] = None) -> None:
    W = 115
    va_s = df["va_global_scale"].iloc[0] if "va_global_scale" in df else float("nan")
    va_t = df["va_global_shift"].iloc[0] if "va_global_shift" in df else float("nan")
    print("\n" + "═" * W)
    print("  DA3METRIC-LARGE — Vehicle-Anchored Scale/Shift Correction — Summary")
    print(f"  Masks evaluated: {len(df):,}   |   Frames: {df['img_id'].nunique():,}")
    if ss_result:
        print(f"  Global scale = {ss_result.scale:.4f}  shift = {ss_result.shift:.4f}  "
              f"anchors = {ss_result.n_inliers}/{ss_result.n_anchors} inliers  "
              f"RMSE_anchor = {ss_result.rmse_m:.3f} m  method = {ss_result.method}")
    print("═" * W)
    hdr = (f"{'Scale Method':<28}"
           f"{'Scale':>10}{'Shift(m)':>10}"
           f"{'RMSE(m)':>10}{'AbsRel':>10}"
           f"{'δ<1.25':>10}{'δ<1.25²':>10}{'δ<1.25³':>10}")
    print(hdr); print("─" * W)
    specs = [
        ("Unscaled",               "unscaled_rmse","unscaled_abs_rel","unscaled_delta1","unscaled_delta2","unscaled_delta3",None,None,False),
        ("Median Scale (oracle)",  "median_rmse",  "median_abs_rel",  "median_delta1",  "median_delta2",  "median_delta3",  "median_scale",None,False),
        ("Mean Scale (oracle)",    "mean_rmse",    "mean_abs_rel",    "mean_delta1",    "mean_delta2",    "mean_delta3",    "mean_scale",  None,False),
        ("Affine Scale (oracle)",  "affine_rmse",  "affine_abs_rel",  "affine_delta1",  "affine_delta2",  "affine_delta3",  "affine_scale","affine_shift",False),
        ("Vehicle-Anchored (ours)","va_rmse",      "va_abs_rel",      "va_delta1",      "va_delta2",      "va_delta3",      None,None,True),
    ]
    for row_spec in specs:
        label, rc, ac, d1c, d2c, d3c, sc, shc, is_va = row_spec
        sv  = df[sc].mean()  if sc   else (va_s if is_va else 1.0)
        shv = df[shc].mean() if shc  else (va_t if is_va else 0.0)
        marker = " ★" if is_va else ""
        print(
            f"  {label:<26}{marker}"
            f"{sv:>10.4f}{shv:>10.4f}"
            f"{df[rc].mean():>10.4f}{df[ac].mean():>10.4f}"
            f"{df[d1c].mean():>10.4f}{df[d2c].mean():>10.4f}{df[d3c].mean():>10.4f}"
        )
    print("═" * W + "\n")
    print("  ★ Vehicle-Anchored: no LiDAR ground truth used — purely from vehicle dimensions + DA3 intrinsics\n")

# ══════════════════════════════════════════════════════════════════════════════
# 15.  MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main() -> None:
    os.makedirs(MAIN_OUTPUT_DIR, exist_ok=True)
    os.makedirs(PLOT_DIR, exist_ok=True)

    # ── Index dataset ──────────────────────────────────────────────────────────
    log.info("─── Indexing KITTI Eigen-Split …")
    val_list = index_kitti(use_train=USE_TRAIN)
    if val_list.empty:
        sys.exit("❌  No frames found. Check KITTI_ROOT and TEST_ROOT paths.")

    # ── Load DA3 depth model ───────────────────────────────────────────────────
    da3_mdl = load_depth_model()

    # ── Pass 1: Anchor collection (or load from cache) ─────────────────────────
    ss_result: Optional[ScaleShiftResult] = None

    if not SKIP_PASS1:
        log.info("─── Pass 1: Collecting vehicle anchors …")
        all_anchors = run_pass1_collect_anchors(val_list, da3_mdl)

        log.info("─── Estimating global scale/shift from %d anchors …", len(all_anchors))
        ss_result = estimate_global_scale_shift(all_anchors)

        # Cache scale/shift to disk
        with open(SCALE_CACHE, "w") as f:
            json.dump({"scale": ss_result.scale, "shift": ss_result.shift,
                       "n_anchors": ss_result.n_anchors, "n_inliers": ss_result.n_inliers,
                       "rmse_m": ss_result.rmse_m, "method": ss_result.method}, f, indent=2)
        log.info("Scale/shift cached → %s", SCALE_CACHE)
    else:
        if not os.path.exists(SCALE_CACHE):
            sys.exit(f"❌  SKIP_PASS1=True but no cache found at {SCALE_CACHE}")
        with open(SCALE_CACHE) as f:
            cache = json.load(f)
        ss_result = ScaleShiftResult(
            scale=cache["scale"], shift=cache["shift"],
            n_anchors=cache.get("n_anchors", 0), n_inliers=cache.get("n_inliers", 0),
            rmse_m=cache.get("rmse_m", float("nan")), method=cache.get("method", "loaded"),
        )
        log.info("Loaded cached scale=%.4f  shift=%.4f", ss_result.scale, ss_result.shift)

    va_scale = ss_result.scale
    va_shift = ss_result.shift
    log.info("Vehicle-Anchored correction: scale=%.4f  shift=%.4f", va_scale, va_shift)

    # ── Pass 2: Evaluation ────────────────────────────────────────────────────
    if not SKIP_INFERENCE:
        if not os.path.exists(OUTPUT_CSV_PATH):
            pd.DataFrame(columns=CSV_COLUMNS).to_csv(OUTPUT_CSV_PATH, index=False)
        log.info("─── Pass 2: Evaluating depth maps …")
        run_pass2_evaluate(val_list, da3_mdl, va_scale, va_shift)
    else:
        log.info("Skipping Pass 2 — using existing CSV at %s", OUTPUT_CSV_PATH)

    # ── Analysis ──────────────────────────────────────────────────────────────
    if not os.path.exists(OUTPUT_CSV_PATH):
        log.warning("No results CSV found. Run Pass 2 first.")
        return

    df = load_results()
    if df.empty:
        log.warning("Results CSV is empty — nothing to analyse.")
        return

    print_summary(df, ss_result)

    if not SKIP_PLOTS:
        generate_all_plots(df)

    log.info("Pipeline complete. Outputs → %s", MAIN_OUTPUT_DIR)


if __name__ == "__main__":
    main()

print('All 13 plot functions defined.')

21:07:37  INFO      ─── Indexing KITTI Eigen-Split …


Indexing Test:   0%|          | 0/652 [00:00<?, ?it/s]

Indexing Train:   0%|          | 0/96 [00:00<?, ?it/s]

21:07:37  INFO      Dataset indexed: 748 frames, 29 drives.
21:07:37  INFO      Loading DA3METRIC-LARGE (depth-anything/DA3METRIC-LARGE) …
21:07:38  INFO      HTTP Request: HEAD https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
21:07:38  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/depth-anything/DA3METRIC-LARGE/4010e39f3634a45bc60553321fb49fb760bd594e/config.json "HTTP/1.1 200 OK"


[INFO ] using MLP layer as FFN


21:07:41  INFO      HTTP Request: HEAD https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/main/model.safetensors "HTTP/1.1 302 Found"
21:07:42  INFO      DA3 depth model ready on cuda.
21:07:42  INFO      ─── Pass 1: Collecting vehicle anchors …


... (117 outputs left out)

22:13:00  INFO      Pass 2 complete → /STORAGE/Arka/a_MDE_Code/2_Corrections/Output/DA3_VehicleAnchored_V5_final/DA3_VA_results.csv
22:13:00  INFO      Generating analysis plots → /STORAGE/Arka/a_MDE_Code/2_Corrections/Output/DA3_VehicleAnchored_V5_final/plots



═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  DA3METRIC-LARGE — Vehicle-Anchored Scale/Shift Correction — Summary
  Masks evaluated: 9,079   |   Frames: 741
  Global scale = 1.0799  shift = 0.2301  anchors = 1672/3337 inliers  RMSE_anchor = 1.251 m  method = ransac_wls
═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Scale Method                     Scale  Shift(m)   RMSE(m)    AbsRel    δ<1.25   δ<1.25²   δ<1.25³
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  Unscaled                      1.0000    0.0000    5.0718    0.1544    0.8560    0.9591    0.9808
  Median Scale (oracle)         1.0594    0.0000    2.7779    0.0501    0.9567    0.9845    0.9931
  Mean Scale (oracle)           1.0489    0.0000    2.5484    0.0546    0.9587    0.9874    0.9954
  Affine Scale (oracle)      

22:13:00  INFO        ✓  00_summary_table.png
22:13:14  INFO        ✓  01_gt_vs_pred_all_scales.png
22:13:33  INFO        ✓  02_gt_vs_rmse_all_scales.png
22:13:51  INFO        ✓  03_angle_vs_rmse_all_scales.png
22:13:52  INFO        ✓  04_scale_comparison_bars.png
22:13:53  INFO        ✓  05_model_vs_lidar_by_category.png
22:13:55  INFO        ✓  06_rmse_distributions.png
22:13:56  INFO        ✓  07_scale_distributions.png
22:13:58  INFO        ✓  08_3d_error_surfaces.png
22:13:59  INFO        ✓  09_depth_bin_analysis.png
22:13:59  INFO        ✓  10_rmse_by_category.png
22:14:10  INFO        ✓  11_scale_vs_depth.png
22:14:11  INFO        ✓  12_vehicle_anchor_quality.png
22:14:11  INFO      All 13 plots saved to /STORAGE/Arka/a_MDE_Code/2_Corrections/Output/DA3_VehicleAnchored_V5_final/plots
22:14:11  INFO      Pipeline complete. Outputs → /STORAGE/Arka/a_MDE_Code/2_Corrections/Output/DA3_VehicleAnchored_V5_final


All 13 plot functions defined.


## Cell 17 · Run Pass 1 — Anchor Collection + Global Scale/Shift

**Skip** this cell (set `SKIP_PASS1 = True` in the config cell) if you have
already run it and the cache file exists at `SCALE_CACHE`.

In [18]:
ss_result: Optional[ScaleShiftResult] = None

# ── Index dataset ──────────────────────────────────────────────────────────────
log.info("Indexing KITTI Eigen-Split …")
val_list = index_kitti(use_train=USE_TRAIN)
assert not val_list.empty, "No frames found — check KITTI_ROOT path."
print(f"Dataset: {len(val_list)} frames, {val_list['drive'].nunique()} drives.")

# ── Load DA3 depth model ───────────────────────────────────────────────────────
da3_mdl = load_depth_model()

if not SKIP_PASS1:
    log.info("Starting Pass 1 …")
    all_anchors = run_pass1_collect_anchors(val_list, da3_mdl)
    print(f"\nTotal vehicle anchors collected: {len(all_anchors)}")

    log.info("Estimating global scale/shift …")
    ss_result = estimate_global_scale_shift(all_anchors)

    # Cache to disk
    with open(SCALE_CACHE, "w") as f:
        json.dump({"scale": ss_result.scale, "shift": ss_result.shift,
                   "n_anchors": ss_result.n_anchors, "n_inliers": ss_result.n_inliers,
                   "rmse_m": ss_result.rmse_m, "method": ss_result.method}, f, indent=2)
    print(f"\nGlobal scale/shift cached → {SCALE_CACHE}")
else:
    assert os.path.exists(SCALE_CACHE), f"Cache not found: {SCALE_CACHE}"
    with open(SCALE_CACHE) as f:
        cache = json.load(f)
    ss_result = ScaleShiftResult(
        scale=cache["scale"], shift=cache["shift"],
        n_anchors=cache.get("n_anchors", 0), n_inliers=cache.get("n_inliers", 0),
        rmse_m=cache.get("rmse_m", float("nan")), method=cache.get("method", "loaded"),
    )
    log.info("Loaded cached scale=%.4f  shift=%.4f", ss_result.scale, ss_result.shift)

print(f"\n{'─'*60}")
print(f"Vehicle-Anchored correction:")
print(f"  scale = {ss_result.scale:.4f}")
print(f"  shift = {ss_result.shift:.4f} m")
print(f"  inliers = {ss_result.n_inliers} / {ss_result.n_anchors}")
print(f"  RMSE (anchors) = {ss_result.rmse_m:.3f} m")
print(f"  method = {ss_result.method}")
print(f"{'─'*60}")

va_scale = ss_result.scale
va_shift = ss_result.shift

22:14:11  INFO      Indexing KITTI Eigen-Split …


Indexing Test:   0%|          | 0/652 [00:00<?, ?it/s]

Indexing Train:   0%|          | 0/96 [00:00<?, ?it/s]

22:14:11  INFO      Dataset indexed: 748 frames, 29 drives.
22:14:11  INFO      Starting Pass 1 …


Dataset: 748 frames, 29 drives.


Pass 1 — Anchor Collection:   0%|          | 0/748 [00:00<?, ?it/s]

22:37:56  INFO      Pass 1 complete: 3337 anchors  (105 frames had no vehicle masks).
22:37:56  INFO      Anchors CSV saved → /STORAGE/Arka/a_MDE_Code/2_Corrections/Output/DA3_VehicleAnchored_V5_final/vehicle_anchors.csv
22:37:56  INFO      Intrinsics CSV saved → /STORAGE/Arka/a_MDE_Code/2_Corrections/Output/DA3_VehicleAnchored_V5_final/sequence_intrinsics.csv
22:37:56  INFO      Estimating global scale/shift …
22:37:56  INFO      Global scale=1.0799  shift=0.2301  inliers=1672/3337  RMSE=1.251m  method=ransac_wls



Total vehicle anchors collected: 3337

Global scale/shift cached → /STORAGE/Arka/a_MDE_Code/2_Corrections/Output/DA3_VehicleAnchored_V5_final/global_scale_shift.json

────────────────────────────────────────────────────────────
Vehicle-Anchored correction:
  scale = 1.0799
  shift = 0.2301 m
  inliers = 1672 / 3337
  RMSE (anchors) = 1.251 m
  method = ransac_wls
────────────────────────────────────────────────────────────


## Cell 18 · Run Pass 2 — Depth Evaluation

Set `SKIP_INFERENCE = True` in the config cell to skip inference and jump
directly to analysis using an existing CSV.

In [19]:
if not SKIP_INFERENCE:
    if not os.path.exists(OUTPUT_CSV_PATH):
        pd.DataFrame(columns=CSV_COLUMNS).to_csv(OUTPUT_CSV_PATH, index=False)
    log.info("Starting Pass 2 …")
    run_pass2_evaluate(val_list, da3_mdl, va_scale, va_shift)
else:
    log.info("Skipping Pass 2 — using existing CSV at %s", OUTPUT_CSV_PATH)

print("Pass 2 complete.")

22:37:56  INFO      Starting Pass 2 …


Pass 2 — Evaluation:   0%|          | 0/748 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Cell 19 · Load Results & Print Summary

In [ ]:
assert os.path.exists(OUTPUT_CSV_PATH), f"Results CSV not found: {OUTPUT_CSV_PATH}"
df = load_results()
print(f"Loaded {len(df):,} rows from {OUTPUT_CSV_PATH}")
print_summary(df, ss_result if 'ss_result' in dir() else None)

Loaded 1,216 rows from /STORAGE/Arka/a_MDE_Code/2_Corrections/Output/DA3_VehicleAnchored_V5/DA3_VA_results.csv

═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  DA3METRIC-LARGE — Vehicle-Anchored Scale/Shift Correction — Summary
  Masks evaluated: 1,216   |   Frames: 48
  Global scale = 1.0376  shift = 0.7828  anchors = 220/351 inliers  RMSE_anchor = 1.246 m  method = ransac_wls
═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Scale Method                     Scale  Shift(m)   RMSE(m)    AbsRel    δ<1.25   δ<1.25²   δ<1.25³
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  Unscaled                      1.0000    0.0000    4.5388    0.1259    0.8809    0.9693    0.9869
  Median Scale (oracle)         1.0535    0.0000    2.5883    0.0459    0.9609    0.9864    0.9939
  Mean Scale (oracle

## Cell 20 · Generate All 13 Analysis Plots

In [ ]:
if not SKIP_PLOTS:
    generate_all_plots(df)
    print(f"\nAll plots saved to: {PLOT_DIR}")
    import glob as _glob
    pngs = sorted(_glob.glob(os.path.join(PLOT_DIR, "*.png")))
    print(f"Generated {len(pngs)} plot files:")
    for p in pngs:
        print(f"  {os.path.basename(p)}")
else:
    print("SKIP_PLOTS=True — skipping plot generation.")

18:35:05  INFO      Generating analysis plots → /STORAGE/Arka/a_MDE_Code/2_Corrections/Output/DA3_VehicleAnchored_V5/plots
18:35:05  INFO        ✓  00_summary_table.png
18:35:06  INFO        ✓  01_gt_vs_pred_all_scales.png
18:35:08  INFO        ✓  02_gt_vs_rmse_all_scales.png
18:35:09  INFO        ✓  03_angle_vs_rmse_all_scales.png
18:35:10  INFO        ✓  04_scale_comparison_bars.png
18:35:10  INFO        ✓  05_model_vs_lidar_by_category.png
18:35:12  INFO        ✓  06_rmse_distributions.png
18:35:13  INFO        ✓  07_scale_distributions.png
18:35:14  INFO        ✓  08_3d_error_surfaces.png
18:35:15  INFO        ✓  09_depth_bin_analysis.png
18:35:15  INFO        ✓  10_rmse_by_category.png
18:35:16  INFO        ✓  11_scale_vs_depth.png
18:35:17  INFO        ✓  12_vehicle_anchor_quality.png
18:35:17  INFO      All 13 plots saved to /STORAGE/Arka/a_MDE_Code/2_Corrections/Output/DA3_VehicleAnchored_V5/plots



All plots saved to: /STORAGE/Arka/a_MDE_Code/2_Corrections/Output/DA3_VehicleAnchored_V5/plots
Generated 13 plot files:
  00_summary_table.png
  01_gt_vs_pred_all_scales.png
  02_gt_vs_rmse_all_scales.png
  03_angle_vs_rmse_all_scales.png
  04_scale_comparison_bars.png
  05_model_vs_lidar_by_category.png
  06_rmse_distributions.png
  07_scale_distributions.png
  08_3d_error_surfaces.png
  09_depth_bin_analysis.png
  10_rmse_by_category.png
  11_scale_vs_depth.png
  12_vehicle_anchor_quality.png
